In [6]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # prevent OpenMP crash on Windows
import pandas as pd
import torch
import torch_geometric
import nilearn
import monai

print("✅ Imports OK")
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("PyG:", torch_geometric.__version__)
print("MONAI:", monai.__version__)
print("Nilearn:", nilearn.__version__)


✅ Imports OK
Torch: 2.9.0+cpu | CUDA available: False
PyG: 2.7.0
MONAI: 1.5.1
Nilearn: 0.12.1


In [21]:




data = pd.read_csv('training_data.csv')
#we don't need first line which contains coloumn names
X_numpy = data[['S0', 'K', 'T', 'sigma', 'r']].values
y_numpy = data[['price']].values
# converting them to tensor
X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
y_tensor = torch.tensor(y_numpy, dtype=torch.float32)


         S0         K         T     sigma         r     price
0  109.0500  116.6140  1.290070  0.315836  0.097763   8.41934
1  106.4620  101.1460  1.629860  0.148158  0.039686   9.65373
2  103.3500   81.2389  1.875040  0.127506  0.080144  26.05690
3   93.4418   93.7010  0.975934  0.464779  0.068633  10.85510
4  102.8250   80.3131  0.752167  0.397235  0.073216  24.63420


In [ ]:
class MinmaxScaling:
    def __init__ (self):
        self.max_xy= None
        self.min_xy = None
        self.difference = None
    def fit(self, tensor):
        self.max_xy = tensor.max(dim = 0).values
        self.min_xy = tensor.min(dim = 0).values
        self.difference = self.max_xy- self.min_xy
    def Scalingdown (self, tensor):
        solution = (tensor-self.min_xy)/self.difference
        return solution
    def Scalingup (self, wannabeinv_tensor):
        solution = (wannabeinv_tensor*self.difference)+self.min_xy
        return solution
        

In [ ]:

def Initialization (X_tensor, y_tensor):
        scaling_X=MinmaxScaling()
        scaling_y = MinmaxScaling()
    
        scaling_X.fit(X_tensor)
        scaling_y.fit(y_tensor)
    
        scaled_up_X = scaling_X.Scalingup(X_tensor)
        scaled_up_y = scaling_y.Scalingup(y_tensor)
        scaled_down_X = scaling_X.Scalingdown(X_tensor)
        scaled_down_y = scaling_y.Scalingdown(y_tensor)
        return scaled_up_X, scaled_up_y, scaled_down_X, scaled_down_y, scaling_X, scaling_y

scaled_up_X, scaled_up_y, scaled_down_X, scaled_down_y, scaling_X, scaling_y = Initialization(X_tensor, y_tensor)
    
T1 = torch.randn( 5, 66, requires_grad=True)
b1 = torch.randn( 1, 66, requires_grad=True)
    
T2 = torch.randn( 66, 10, requires_grad=True)
b2 = torch.randn( 1, 10, requires_grad=True)
    
T3 = torch.randn( 10, 1, requires_grad=True)
b3 =  torch.randn( 1, 1, requires_grad=True)


    




epochs = 1000
learning_rate = 0.001
scaled_up_X, scaled_up_y, scaled_down_X, scaled_down_y, scaling_X, scaling_y = Initialization(X_tensor, y_tensor) #scaled_up_X-nek nincs szerepe
for i in range (epochs):
    T1_Relu =torch.relu(scaled_down_X@T1 + b1)
    T2_Relu = torch.relu(T1_Relu @ T2+ b2)
    T3_Relu = torch.relu(T2_Relu@T3 + b3)
   
    y_pred_scaled_down = T3_Relu
    
    loss = torch.mean((scaled_down_y - y_pred_scaled_down)**2)

    loss.backward()   #This function calculates T1.grad, b1.grad.. etc... Using chain rule : (F(G(x)))' = F'(G(x))*G'(x) so we get in this case
                # so compound function's derivative is the product of each function's derivative

    with torch.no_grad()
        T1 -= learning_rate * T1.grad
        b1 -= learning_rate * b1.grad
        T2 -= learning_rate * T2.grad
        b2 -= learning_rate * b2.grad
        T3 -= learning_rate * T3.grad
        b3 -= learning_rate * b3.grad

        T1.grad.zero_()  
        b1.grad.zero_()
        T2.grad.zero_()
        b2.grad.zero_()
        T3.grad.zero_()
        b3.grad.zero_()
# ez a resze ai

    if i % 100 == 0:
        print(f"Epoch {i}: Loss = {loss.item():.6f}")


print("\n--- Tanítás kész! ---")
with torch.no_grad():
    y_predicted_dollar = caling_y.Scalingup(y_pred_scaled)

print(f"Első becslés (Dollárban): {y_predicted_dollar[0].item():.2f}")
print(f"Első valós ár (Dollárban): {y_tensor[0].item():.2f}")
